In [ ]:
# General imports
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate

# cloelib imports
from cloelib.cosmology.camb_cosmology import CAMBBackground
from cloelib.observables.photo import ShearTracer, PositionsTracer
from cloelib.summary_statistics.angular_two_point import AngularTwoPoint
from cloelib.observables.photo.spectrum_engine import SpectraBank

import euclidlib as el
from mpl_toolkits.axes_grid1 import make_axes_locatable

import pandas as pd
import xarray as xr

from tabulated_pk_interpolator import TabulatedMatterPowerInterpolator
from tabulated_perturbations import TabulatedPerturbations

# Plot style
import seaborn as sns
sns.set_theme(style="ticks")
sns.set_palette(sns.color_palette("Paired"))

plt.rc('xtick',labelsize=20)
plt.rc('ytick',labelsize=20)
plt.rc('font',size=20)
plt.rc('axes', titlesize=25)
plt.rc('axes', labelsize=20)
plt.rc('lines', linewidth=3)
plt.rc('lines', markersize=6)
plt.rc('legend', fontsize=14)

## Cosmology: Background

This code snippet initializes and computes the cosmological background.

In [ ]:
zs = np.linspace(1e-4, 3, 1000)

# The arguments of the Background functions follow the cosmology.API
background = CAMBBackground(H0=70.0, 
                            Omega_cdm0=0.248684,
                            Omega_b0=0.05, 
                            w0=-1, 
                            wa=0, 
                            Omega_k0 = 0.0, 
                            ns = 0.96, 
                            As = 2e-9,
                            mnu = 0.06,
                            gamma_MG = 0.545,
                            N_mnu = 1)

## Download and Read the Data

In this step, we use the `euclidlib` library to read the data, which follows . `euclidlib` provides the necessary functions and utilities to access and handle data from the Euclid mission, allowing us to seamlessly retrieve, process, and analyze the large datasets generated by the mission. 

For this tutorial, we use example synthetic data stored at https://zenodo.org/communities/cloe-org


In [ ]:
# Execute this cell to download the data
data_downloaded = True

if data_downloaded == False:
    
    import requests
    
    # URLs of the files to be downloaded
    urls = {
        'nz_example.fits': 'https://zenodo.org/records/15092862/files/nz_example.fits',
        'mixmats_example.fits': 'https://zenodo.org/records/17191365/files/example-mixmats.fits',
        'cls_example.fits': 'https://zenodo.org/records/17191365/files/example-spectra.fits'
    }
    
    # Function to download a file
    def download_file(url, filename):
        response = requests.get(url)
        if response.status_code == 200:
            with open(filename, 'wb') as f:
                f.write(response.content)
            print(f'{filename} downloaded successfully')
        else:
            print(f'Failed to download {filename}. Status code: {response.status_code}')
    
    # Download all files
    for filename, url in urls.items():
        download_file(url, filename)

In [ ]:
# Read the data using euclidlib v2025.w
# Remember, only pip install euclidlib necessary!

z_nz, nz_example = el.phz.redshift_distributions('nz_example.fits')
cls_example = el.le3.pk_wl.angular_power_spectra('cls_example.fits')
mixmats_example = el.le3.pk_wl.mixing_matrices('mixmats_example.fits')

## Plot the galaxy redshift distribution $n(z)$

We use a synthetic galaxy redshift bin distribution that contains 6 redshift bins. We assume that both sources and lenses follow the same distributions.

In [ ]:
# Function to normalize dndz
def normalize_dndz(nz_example, z_nz):
    normalized = {}
    for key in nz_example:
        normalized[key] = nz_example[key] / integrate.trapezoid(nz_example[key], z_nz)
    return normalized

# Normalize both dndz_pos and dndz_she
dndz_pos_norm = normalize_dndz(nz_example, z_nz)
dndz_she_norm = normalize_dndz(nz_example, z_nz)

# Function to resample the normalized dndz
def resample_dndz(nz_example, z_nz, myz):
    my_dndz = np.vstack(list(nz_example.values()))
    my_dndz_norm = np.zeros([len(nz_example), len(myz)])
    for i in range(len(nz_example)):
        my_dndz_norm[i, :] = np.interp(myz, z_nz, my_dndz[i, :])
    return my_dndz_norm

# Resampling the normalized dndz with 100 values
my_dndz_pos_norm = resample_dndz(dndz_pos_norm, z_nz, zs)
my_dndz_she_norm = resample_dndz(dndz_she_norm, z_nz, zs)

# Plotting dndz
fig, axs = plt.subplots(2, 1, figsize=(10, 10))

# Customize tick parameters for both axes
for ax in axs.flatten():
    ax.tick_params(axis='both', which='both', direction='in')

# Plot positions n(z)
for i, (key, color) in enumerate(zip(dndz_pos_norm.keys(), sns.color_palette("rocket", len(dndz_pos_norm)))):
    axs[0].plot(z_nz, dndz_pos_norm[key], label=f'$n_{{{i+1}}}$', linewidth=2, color=color)

# Plot shear n(z)
for j, (key, color) in enumerate(zip(dndz_she_norm.keys(), sns.color_palette("mako", len(dndz_she_norm)))):
    axs[1].plot(z_nz, dndz_she_norm[key], label=f'$n_{{{j+1}}}$', linewidth=2, color=color)

# Set labels and legends
axs[0].set_xlabel(r'$z$')
axs[0].set_ylabel(r'Positions $n(z)$')
axs[0].set_xlim(0, 4)
axs[0].legend(frameon=False, ncol=2)

axs[1].set_xlabel(r'$z$')
axs[1].set_ylabel(r'Shear $n(z)$')
axs[1].set_xlim(0, 4)
axs[1].legend(frameon=False, ncol=2)

plt.tight_layout()
plt.show()

## Construct perturbations directly from external linear and nonlinear power spectra

### In this example, the external power spectra come from CosmoSIS. 
CosmoSIS uses h-based units (h/Mpc and (Mpc/h)^3), so we convert to physical units (Mpc^-1, Mpc^3) before passing them in.

In [ ]:
# Local path to external output files
input_path_external_data = '/Users/your_local_path/cosmosis_data/'
input_path_external_matter_power_lin = input_path_external_data + 'matter_power_lin/'
input_path_external_matter_power_nl = input_path_external_data + 'matter_power_nl/'
z_external = np.loadtxt(input_path_external_matter_power_lin + 'z.txt')
k_external = np.loadtxt(input_path_external_matter_power_lin + 'k_h.txt')*background.h
p_k_lin_external = np.loadtxt(input_path_external_matter_power_lin + 'p_k.txt')/(background.h**3)
p_k_nl_external = np.loadtxt(input_path_external_matter_power_nl + 'p_k.txt')/(background.h**3)

In [ ]:
p_k_lin_external_interp = TabulatedMatterPowerInterpolator(z_external, k_external, p_k_lin_external)
p_k_nl_external_interp = TabulatedMatterPowerInterpolator(z_external, k_external, p_k_nl_external)

In [ ]:
# Perturbations built directly from the external matter_power_lin/nl tables
# loaded above, instead of computed by CAMB - satisfies the same
# cloelib.cosmology.cosmology.Perturbations protocol as
# CAMBLinearPerturbations/CAMBNonLinearPerturbations.
linear_perturbations_external = TabulatedPerturbations(
    background, z_external, k_external, p_k_lin_external,
)
nonlinear_perturbations_external = TabulatedPerturbations(
    background, z_external, k_external, p_k_nl_external,
    linearperturbations=linear_perturbations_external,
)

## Observables: shear and position tracers 

We compute tracers (window functions) for both shear and galaxy positions observations, to be later combined in the computation of the summary statistics.

In [ ]:
# Select the perturbations object you want to use
perturbations = nonlinear_perturbations_external

# Define the tracers
tracer_pos = PositionsTracer(perturbations=perturbations, 
                             dndz=my_dndz_pos_norm,
                             z = zs,
                             galaxy_bias_model='per_bin',
                             nuisance_params={'b1_photo_bin0': 1.14, 'b1_photo_bin1': 1.20, 
                                              'b1_photo_bin2': 1.32, 'b1_photo_bin3': 1.42,
                                              'b1_photo_bin4': 1.52, 'b1_photo_bin5': 1.94,
                                              'magnification_bias_1': 0.0, 'magnification_bias_2': 0.0,
                                              'magnification_bias_3': 0.0, 'magnification_bias_4': 0.0,
                                              'magnification_bias_5': 0.0, 'magnification_bias_6': 0.0,
                                              'dz_pos_1': 0.0, 'dz_pos_2': 0.0,
                                              'dz_pos_3': 0.0, 'dz_pos_4': 0.0,
                                              'dz_pos_5': 0.0, 'dz_pos_6': 0.0,
                                              'width_pos_1': 1.0, 'width_pos_2': 1.0,
                                              'width_pos_3': 1.0, 'width_pos_4': 1.0,
                                              'width_pos_5': 1.0, 'width_pos_6': 1.0})

from cloelib.observables.photo.shear import PBJTATTLoopComputer

# Build a ShearTracer using the TATT intrinsic-alignment model. 
# A1/C_IA/eta1 match the NLA nuisance params; A2/b_TA/eta2/z0
# are the zTATT fiducial values from Table 3 of the reference paper.
# FAST-PT's one-loop integrals need the *linear* Pk - pass
# linear_perturbations_external explicitly (don't pass `perturbations`
# itself: CAMBNonLinearPerturbations-style backends without a
# `.linearperturbations` attribute would silently feed FAST-PT the
# *nonlinear* Pk instead, which is physically wrong; TabulatedPerturbations
# doesn't have that gap since we set `linearperturbations=` above, but
# being explicit here avoids relying on that fallback).
nuisance_shear_tatt = {'AIA': 1.72, 'CIA': 0.013873073650776856, 'EtaIA': -0.41,
                        'A2IA': 0.40, 'bTA': -0.83, 'Eta2IA': 2.69, 'z0IA': 0.62,
                        'multiplicative_bias_1': 0.0, 'multiplicative_bias_2': 0.0,
                        'multiplicative_bias_3': 0.0, 'multiplicative_bias_4': 0.0,
                        'multiplicative_bias_5': 0.0, 'multiplicative_bias_6': 0.0,
                        'dz_shear_1': 0.0, 'dz_shear_2': 0.0,
                        'dz_shear_3': 0.0, 'dz_shear_4': 0.0,
                        'dz_shear_5': 0.0, 'dz_shear_6': 0.0,
                        'width_shear_1': 1.0, 'width_shear_2': 1.0,
                        'width_shear_3': 1.0, 'width_shear_4': 1.0,
                        'width_shear_5': 1.0, 'width_shear_6': 1.0}

tracer_she_tatt = ShearTracer(perturbations=perturbations,
                               dndz=my_dndz_she_norm,
                               z=zs,
                               nuisance_params=nuisance_shear_tatt,
                               ia_model="TATT",
                               tatt_loop_computer=PBJTATTLoopComputer(linear_perturbations_external))

tracer_she_lensing = ShearTracer(perturbations=perturbations,
                               dndz=my_dndz_she_norm,
                               z=zs,
                               nuisance_params=nuisance_shear_tatt)

print(f"TATT: tracer_she_tatt.ia -> {type(tracer_she_tatt.ia).__name__}  "
      f"(loop_computer={type(tracer_she_tatt.ia._loop_computer).__name__})")

## Compare intrinsic-intrinsic and delta-intrinsic power spectrum from TATT between cloelib and CosmoSIS

### CosmoSIS power spectra

In [ ]:
input_path_external_intrinsic_power_tatt = input_path_external_data + 'intrinsic_power/'
input_path_external_matter_intrinsic_power_tatt = input_path_external_data + 'matter_intrinsic_power/'
p_k_ii_tatt_external = np.loadtxt(input_path_external_intrinsic_power_tatt + 'p_k.txt')/(background.h**3)
p_k_i_delta_tatt_external = np.loadtxt(input_path_external_matter_intrinsic_power_tatt + 'p_k.txt')/(background.h**3)

p_k_ii_tatt_external_interp = TabulatedMatterPowerInterpolator(z_external, k_external, p_k_ii_tatt_external, log_pk=False)
p_k_i_delta_tatt_external_interp = TabulatedMatterPowerInterpolator(z_external, k_external, p_k_i_delta_tatt_external, log_pk=False)

### cloelib power spectra

In [ ]:
matter_pk = perturbations.matter_power_spectrum(perturbations.z, perturbations.k)
bank = SpectraBank(matter_pk, perturbations.k, perturbations.z)
P_I_I_TATT = tracer_she_tatt.ia.get_effective_pk(tracer_she_tatt.ia, bank)
P_I_delta_TATT = tracer_she_tatt.ia.get_effective_pk(tracer_she_tatt.lensing, bank)

P_I_I_TATT_interp = TabulatedMatterPowerInterpolator(
    perturbations.z, perturbations.k, P_I_I_TATT, log_pk=False
)
P_I_delta_TATT_interp = TabulatedMatterPowerInterpolator(
    perturbations.z, perturbations.k, P_I_delta_TATT, log_pk=False
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (10, 5))
ax1.plot(k_external, p_k_ii_tatt_external_interp(zs = 1, ks = k_external)[0], ls = '-', color = 'r', lw = 1, label = 'CosmoSIS')
ax1.plot(k_external, P_I_I_TATT_interp(zs = 1, ks = k_external)[0], ls = '-', color = 'b', lw = 1, label = 'cloelib')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_ylabel(r'$P_{\rm{II}}(k)$', fontsize=16)
ax1.legend(loc = 'upper right')
ax1.tick_params(axis='both', which='major', labelsize=12)
ax1.tick_params(axis='both', which='minor', labelsize=12)
ax1.tick_params(axis='x', which='both', bottom = False, top = False, labelbottom = False, labelsize=12)

divider = make_axes_locatable(ax1)
axShallow = divider.append_axes("bottom", size="50%", pad=0.1, sharex=ax1)

axShallow.semilogx(k_external, 100*((P_I_I_TATT_interp(zs = 1, ks = k_external)[0]/p_k_ii_tatt_external_interp(zs = 1, ks = k_external)[0])-1), '--', lw = 3, c='black')
axShallow.set_xlabel(r'$k$ $[{Mpc}^{-1}]$', fontsize=16)
ax1.set_xlim(0.001, 10)
axShallow.set_xlim(0.001, 10)
axShallow.fill_between(k_external, -0.2, 0.2, color = 'k', alpha = 0.1)
axShallow.fill_between(k_external, -0.1, 0.1, color = 'k', alpha = 0.3)
axShallow.set_ylabel(r'$\frac{\mathrm{cloelib}}{\mathrm{CosmoSIS}}$ - 1 [%]', fontsize=16)
axShallow.tick_params(axis='both', which='major', labelsize=12)
axShallow.tick_params(axis='both', which='minor', labelsize=12)

ax2.plot(k_external, p_k_i_delta_tatt_external_interp(zs = 1, ks = k_external)[0], ls = '-', color = 'r', lw = 1, label = 'CosmoSIS')
ax2.plot(k_external, P_I_delta_TATT_interp(zs = 1, ks = k_external)[0], ls = '-', color = 'b', lw = 1, label = 'cloelib')
ax2.set_xscale('log')
ax2.set_ylabel(r'$P_{\delta\rm{I}}(k)$', fontsize=16)
ax2.legend(loc = 'upper right')
ax2.tick_params(axis='both', which='major', labelsize=12)
ax2.tick_params(axis='both', which='minor', labelsize=12)
ax2.tick_params(axis='x', which='both', bottom = False, top = False, labelbottom = False, labelsize=12)

divider = make_axes_locatable(ax2)
axShallow = divider.append_axes("bottom", size="50%", pad=0.1, sharex=ax2)

axShallow.semilogx(k_external, 100*((P_I_delta_TATT_interp(zs = 1, ks = k_external)[0]/p_k_i_delta_tatt_external_interp(zs = 1, ks = k_external)[0])-1), '--', lw = 3, c='black')
axShallow.set_xlabel(r'$k$ $[{Mpc}^{-1}]$', fontsize=16)
axShallow.set_xlim(0.001, 10)
axShallow.fill_between(k_external, -0.2, 0.2, color = 'k', alpha = 0.1)
axShallow.fill_between(k_external, -0.1, 0.1, color = 'k', alpha = 0.3)
axShallow.set_ylabel(r'$\frac{\mathrm{cloelib}}{\mathrm{CosmoSIS}}$ - 1 [%]', fontsize=16)
axShallow.tick_params(axis='both', which='major', labelsize=12)
axShallow.tick_params(axis='both', which='minor', labelsize=12)
plt.tight_layout()
plt.show()

## Two-point statistics

### CosmoSIS theoretical Predictions: $C_\ell$

We read angular power spectra from CosmoSIS.

In [ ]:
input_path_external_shear_cl = input_path_external_data + 'shear_cl/'
ells_external = pd.read_csv(input_path_external_shear_cl + 'ell.txt', skiprows=1, header=None).values[:, 0]
# Limber prefactor missing in CosmoSIS
lf_dl=(ells_external+0.5)**2/np.sqrt((ells_external+2.)*(ells_external+1.)*ells_external*(ells_external-1.))

#Shear Cl
coords = {'bin_z1': np.arange(1, 7), 'bin_z2':np.arange(1, 7), 'l': np.squeeze(ells_external)}
dims = ['bin_z1', 'bin_z2', 'l']
shear_cl_external = xr.DataArray(np.zeros((6, 6, len(np.squeeze(ells_external)))), coords=coords, dims=dims)
for i in range(6):
    for j in range(6):
        if i>=j:
            shear_cl_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_shear_cl + 'bin_{}_{}.txt'.format(i+1, j+1), sep = ' ', skiprows=1, header=None)).values*(lf_dl**-2)
        else:
            shear_cl_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_shear_cl + 'bin_{}_{}.txt'.format(j+1, i+1), sep = ' ', skiprows=1, header=None)).values*(lf_dl**-2)

input_path_external_shear_cl_GG = input_path_external_data + 'shear_cl_gg/'
coords = {'bin_z1': np.arange(1, 7), 'bin_z2':np.arange(1, 7), 'l': np.squeeze(ells_external)}
dims = ['bin_z1', 'bin_z2', 'l']
shear_cl_GG_external = xr.DataArray(np.zeros((6, 6, len(np.squeeze(ells_external)))), coords=coords, dims=dims)
for i in range(6):
    for j in range(6):
        if i>=j:
            shear_cl_GG_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_shear_cl_GG + 'bin_{}_{}.txt'.format(i+1, j+1), sep = ' ', skiprows=1, header=None)).values*(lf_dl**-2)
        else:
            shear_cl_GG_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_shear_cl_GG + 'bin_{}_{}.txt'.format(j+1, i+1), sep = ' ', skiprows=1, header=None)).values*(lf_dl**-2)

input_path_external_shear_cl_II = input_path_external_data + 'shear_cl_ii/'
coords = {'bin_z1': np.arange(1, 7), 'bin_z2':np.arange(1, 7), 'l': np.squeeze(ells_external)}
dims = ['bin_z1', 'bin_z2', 'l']
shear_cl_II_external = xr.DataArray(np.zeros((6, 6, len(np.squeeze(ells_external)))), coords=coords, dims=dims)
for i in range(6):
    for j in range(6):
        if i>=j:
            shear_cl_II_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_shear_cl_II + 'bin_{}_{}.txt'.format(i+1, j+1), sep = ' ', skiprows=1, header=None)).values*(lf_dl**-2)
        else:
            shear_cl_II_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_shear_cl_II + 'bin_{}_{}.txt'.format(j+1, i+1), sep = ' ', skiprows=1, header=None)).values*(lf_dl**-2)

input_path_external_shear_cl_GI = input_path_external_data + 'shear_cl_gi/'
coords = {'bin_z1': np.arange(1, 7), 'bin_z2':np.arange(1, 7), 'l': np.squeeze(ells_external)}
dims = ['bin_z1', 'bin_z2', 'l']
shear_cl_GI_external = xr.DataArray(np.zeros((6, 6, len(np.squeeze(ells_external)))), coords=coords, dims=dims)
for i in range(6):
    for j in range(6):
        shear_cl_GI_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_shear_cl_GI + 'bin_{}_{}.txt'.format(i+1, j+1), sep = ' ', skiprows=1, header=None)).values*(lf_dl**-2)

#Galaxy shear Cl
input_path_external_galaxy_shear_cl = input_path_external_data + 'galaxy_shear_cl/'
coords = {'bin_z1': np.arange(1, 7), 'bin_z2':np.arange(1, 7), 'l': np.squeeze(ells_external)}
dims = ['bin_z1', 'bin_z2', 'l']
galaxy_shear_cl_external = xr.DataArray(np.zeros((6, 6, len(np.squeeze(ells_external)))), coords=coords, dims=dims)
for i in range(6):
    for j in range(6):
        galaxy_shear_cl_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_galaxy_shear_cl + 'bin_{}_{}.txt'.format(i+1, j+1), sep = ' ', skiprows=1, header=None)).values*(lf_dl**-1)

input_path_external_galaxy_shear_gI_cl = input_path_external_data + 'galaxy_intrinsic_cl/'
coords = {'bin_z1': np.arange(1, 7), 'bin_z2':np.arange(1, 7), 'l': np.squeeze(ells_external)}
dims = ['bin_z1', 'bin_z2', 'l']
galaxy_shear_gI_cl_external = xr.DataArray(np.zeros((6, 6, len(np.squeeze(ells_external)))), coords=coords, dims=dims)
for i in range(6):
    for j in range(6):
        galaxy_shear_gI_cl_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_galaxy_shear_gI_cl + 'bin_{}_{}.txt'.format(i+1, j+1), sep = ' ', skiprows=1, header=None)).values*(lf_dl**-1)

#Galaxy clustering Cl
input_path_external_galaxy_cl = input_path_external_data + 'galaxy_cl/'
coords = {'bin_z1': np.arange(1, 7), 'bin_z2':np.arange(1, 7), 'l': np.squeeze(ells_external)}
dims = ['bin_z1', 'bin_z2', 'l']
galaxy_cl_external = xr.DataArray(np.zeros((6, 6, len(np.squeeze(ells_external)))), coords=coords, dims=dims)
for i in range(6):
    for j in range(6):
        if i>=j:
            galaxy_cl_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_galaxy_cl + 'bin_{}_{}.txt'.format(i+1, j+1), sep = ' ', skiprows=1, header=None)).values
        else:
            galaxy_cl_external[i, j, :] = np.squeeze(pd.read_csv(input_path_external_galaxy_cl + 'bin_{}_{}.txt'.format(j+1, i+1), sep = ' ', skiprows=1, header=None)).values

### cloelib theoretical Predictions: $C_\ell$

We compute angular power spectra initialising the corresponding `cloelib` class.

In [ ]:
nl = 100
ells = np.logspace(1., np.log10(3000), nl)

# Total shear = GG + IG + GI + II

twopoint_sheshe_total_tatt = AngularTwoPoint(tracer_she_tatt, tracer_she_tatt)
cells_sheshe_total_tatt_cloelib = twopoint_sheshe_total_tatt.get_Cl(ells_external, 0, perturbations.k)

twopoint_sheshe_GG_tatt = AngularTwoPoint(tracer_she_lensing, tracer_she_lensing)
cells_sheshe_GG_tatt_cloelib = twopoint_sheshe_GG_tatt.get_Cl(ells_external, 0, perturbations.k)

twopoint_sheshe_GI_tatt = AngularTwoPoint(tracer_she_lensing, tracer_she_tatt)
cells_sheshe_GI_tatt_cloelib = twopoint_sheshe_GI_tatt.get_Cl(ells_external, 0, perturbations.k)

twopoint_sheshe_IG_tatt = AngularTwoPoint(tracer_she_tatt, tracer_she_lensing)
cells_sheshe_IG_tatt_cloelib = twopoint_sheshe_IG_tatt.get_Cl(ells_external, 0, perturbations.k)

# Total galaxy shear = gG + gI

twopoint_posshe_total_tatt = AngularTwoPoint(tracer_pos, tracer_she_tatt)
cells_posshe_total_tatt_cloelib = twopoint_posshe_total_tatt.get_Cl(ells_external, 0, perturbations.k)

twopoint_posshe_gG_tatt = AngularTwoPoint(tracer_pos, tracer_she_lensing)
cells_posshe_gG_tatt_cloelib = twopoint_posshe_gG_tatt.get_Cl(ells_external, 0, perturbations.k)

# Total galaxy clustering = gg

twopoint_pospos_total = AngularTwoPoint(tracer_pos, tracer_pos)
cells_pospos_total_cloelib = twopoint_pospos_total.get_Cl(ells_external, 0, perturbations.k)

## Shear Cl comparison

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10), dpi=200)
fig.suptitle(r'WL $C^{ij}_{\ell}$ (TATT)', fontsize=50)

k = 0
for iz_1 in range(1, 6+1):
    ax1 = axs[k//3, k%3]
    for iz_2 in range(iz_1, 6+1):
        if iz_1 == iz_2:
            ls_i = '-'
        else:
            ls_i = '--'
        cl_cloelib = np.asarray(cells_sheshe_total_tatt_cloelib[("SHE", "SHE", iz_1, iz_2)][0, 0])
        ax1.semilogx(ells_external, 100*((cl_cloelib/(shear_cl_external.sel(bin_z1=iz_1, bin_z2=iz_2, l=ells_external)))-1), color='black', ls=ls_i, lw=3)
    ax1.set_xlabel(r'$\ell$', fontsize=36)
    ax1.fill_between(ells_external, -0.2, 0.2, color = 'k', alpha = 0.1)
    ax1.fill_between(ells_external, -0.1, 0.1, color = 'k', alpha = 0.3)
    if k%3 == 0:
        ax1.set_ylabel(r'$\frac{\mathrm{cloelib}}{\mathrm{CosmoSIS}}$ - 1 [%]', fontsize=26)
    ax1.tick_params(axis='both', which='major', labelsize=20)
    ax1.tick_params(axis='both', which='minor', labelsize=20)
    ax1.set_title('bin i={:d}'.format(iz_1), fontsize=26)    
    k = k + 1
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10), dpi=200)
fig.suptitle(r'WL $C^{ij}_{\ell}$ (TATT)  (Only intrinsic intrinsic contribution)', fontsize=50)

k = 0
for iz_1 in range(1, 6+1):
    ax1 = axs[k//3, k%3]
    for iz_2 in range(iz_1, 6+1):
        if iz_1 == iz_2:
            ls_i = '-'
            cl_cloelib = np.asarray(cells_sheshe_total_tatt_cloelib[("SHE", "SHE", iz_1, iz_2)][0, 0]-cells_sheshe_GI_tatt_cloelib[("SHE", "SHE", iz_1, iz_2)][0, 0]-cells_sheshe_IG_tatt_cloelib[("SHE", "SHE", iz_1, iz_2)][0, 0]+ cells_sheshe_GG_tatt_cloelib[("SHE", "SHE", iz_1, iz_2)][0, 0])
            ax1.semilogx(ells_external, 100*((cl_cloelib/(shear_cl_II_external.sel(bin_z1=iz_1, bin_z2=iz_2, l=ells_external)))-1), color='black', ls=ls_i, lw=3)
    ax1.set_xlabel(r'$\ell$', fontsize=36)
    ax1.fill_between(ells_external, -0.2, 0.2, color = 'k', alpha = 0.1)
    ax1.fill_between(ells_external, -0.1, 0.1, color = 'k', alpha = 0.3)
    if k%3 == 0:
        ax1.set_ylabel(r'$\frac{\mathrm{cloelib}}{\mathrm{CosmoSIS}}$ - 1 [%]', fontsize=26)
    ax1.tick_params(axis='both', which='major', labelsize=20)
    ax1.tick_params(axis='both', which='minor', labelsize=20)
    ax1.set_title('bin i={:d}'.format(iz_1), fontsize=26)    
    k = k + 1
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10), dpi=200)
fig.suptitle(r'WL $C^{ij}_{\ell}$ (TATT)  (Only shear intrinsic contribution)', fontsize=50)

k = 0
for iz_1 in range(1, 6+1):
    ax1 = axs[k//3, k%3]
    for iz_2 in range(iz_1, 6+1):
        if iz_1 == iz_2:
            ls_i = '-'
            cl_cloelib = np.asarray(cells_sheshe_GI_tatt_cloelib[("SHE", "SHE", iz_1, iz_2)][0, 0]-cells_sheshe_GG_tatt_cloelib[("SHE", "SHE", iz_1, iz_2)][0, 0])
            ax1.semilogx(ells_external, 100*((cl_cloelib/(shear_cl_GI_external.sel(bin_z1=iz_1, bin_z2=iz_2, l=ells_external)))-1), color='black', ls=ls_i, lw=3)
    ax1.set_xlabel(r'$\ell$', fontsize=36)
    ax1.fill_between(ells_external, -0.2, 0.2, color = 'k', alpha = 0.1)
    ax1.fill_between(ells_external, -0.1, 0.1, color = 'k', alpha = 0.3)
    if k%3 == 0:
        ax1.set_ylabel(r'$\frac{\mathrm{cloelib}}{\mathrm{CosmoSIS}}$ - 1 [%]', fontsize=26)
    ax1.tick_params(axis='both', which='major', labelsize=20)
    ax1.tick_params(axis='both', which='minor', labelsize=20)
    ax1.set_title('bin i={:d}'.format(iz_1), fontsize=26)    
    k = k + 1
plt.tight_layout()
plt.show()

## Galaxy-shear Cl comparison

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10), dpi=200)
fig.suptitle(r'WLxGC $C^{ij}_{\ell}$ (TATT)', fontsize=50)

k = 0
for iz_1 in range(1, 6+1):
    ax1 = axs[k//3, k%3]
    for iz_2 in range(iz_1, 6+1):
        if iz_1 == iz_2:
            ls_i = '-'
        else:
            ls_i = '--'
        cl_cloelib = np.asarray(cells_posshe_total_tatt_cloelib[("POS", "SHE", iz_1, iz_2)][0])
        ax1.semilogx(ells_external, 100*((cl_cloelib/(galaxy_shear_cl_external.sel(bin_z1=iz_1, bin_z2=iz_2, l=ells_external)))-1), color='black', ls=ls_i, lw=3)
    ax1.set_xlabel(r'$\ell$', fontsize=36)
    ax1.set_ylim(-0.5, 0.5)
    ax1.fill_between(ells_external, -0.2, 0.2, color = 'k', alpha = 0.1)
    ax1.fill_between(ells_external, -0.1, 0.1, color = 'k', alpha = 0.3)
    if k%3 == 0:
        ax1.set_ylabel(r'$\frac{\mathrm{cloelib}}{\mathrm{CosmoSIS}}$ - 1 [%]', fontsize=26)
    ax1.tick_params(axis='both', which='major', labelsize=20)
    ax1.tick_params(axis='both', which='minor', labelsize=20)
    ax1.set_title('bin i={:d}'.format(iz_1), fontsize=26)    
    k = k + 1
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10), dpi=200)
fig.suptitle(r'WLxGC $C^{ij}_{\ell}$ (TATT)  (Only galaxy intrinsic contributions)', fontsize=50)

k = 0
for iz_1 in range(1, 6+1):
    ax1 = axs[k//3, k%3]
    for iz_2 in range(iz_1, 6+1):
        if iz_1 == iz_2:
            ls_i = '-'
            cl_cloelib = np.asarray(cells_posshe_total_tatt_cloelib[("POS", "SHE", iz_1, iz_2)][0]-cells_posshe_gG_tatt_cloelib[("POS", "SHE", iz_1, iz_2)][0])
            ax1.semilogx(ells_external, 100*((cl_cloelib/(galaxy_shear_gI_cl_external.sel(bin_z1=iz_1, bin_z2=iz_2, l=ells_external)))-1), color='black', ls=ls_i, lw=3)
    ax1.set_xlabel(r'$\ell$', fontsize=36)
    ax1.set_ylim(-0.5, 0.5)
    ax1.fill_between(ells_external, -0.2, 0.2, color = 'k', alpha = 0.1)
    ax1.fill_between(ells_external, -0.1, 0.1, color = 'k', alpha = 0.3)
    if k%3 == 0:
        ax1.set_ylabel(r'$\frac{\mathrm{cloelib}}{\mathrm{CosmoSIS}}$ - 1 [%]', fontsize=26)
    ax1.tick_params(axis='both', which='major', labelsize=20)
    ax1.tick_params(axis='both', which='minor', labelsize=20)
    ax1.set_title('bin i={:d}'.format(iz_1), fontsize=26)    
    k = k + 1
plt.tight_layout()
plt.show()

## Galaxy clustering Cl comparison

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10), dpi=200)
fig.suptitle(r'GC $C^{ij}_{\ell}$', fontsize=50)

k = 0
for iz_1 in range(1, 6+1):
    ax1 = axs[k//3, k%3]
    for iz_2 in range(iz_1, 6+1):
        if iz_1 == iz_2:
            ls_i = '-'
            cl_cloelib = np.asarray(cells_pospos_total_cloelib[("POS", "POS", iz_1, iz_2)])
            ax1.semilogx(ells_external, 100*((cl_cloelib/(galaxy_cl_external.sel(bin_z1=iz_1, bin_z2=iz_2, l=ells_external)))-1), color='black', ls=ls_i, lw=3)
    ax1.set_xlabel(r'$\ell$', fontsize=36)
    ax1.fill_between(ells_external, -0.2, 0.2, color = 'k', alpha = 0.1)
    ax1.fill_between(ells_external, -0.1, 0.1, color = 'k', alpha = 0.3)
    if k%3 == 0:
        ax1.set_ylabel(r'$\frac{\mathrm{cloelib}}{\mathrm{CosmoSIS}}$ - 1 [%]', fontsize=26)
    ax1.tick_params(axis='both', which='major', labelsize=20)
    ax1.tick_params(axis='both', which='minor', labelsize=20)
    ax1.set_title('bin i={:d}'.format(iz_1), fontsize=26)    
    k = k + 1
plt.tight_layout()
plt.show()